# AI-Based Environmental Pollution Forecasting System

# Notebook 01 : Data Preparation

## Objective

This notebook performs the complete data preparation pipeline for the Environmental Pollution Forecasting System.

The notebook is responsible for:

- Loading raw datasets
- Merging multiple files
- Standardizing schema
- Cleaning missing and duplicate values
- Correcting data types
- Creating a unified datetime column
- Validating data quality
- Saving a clean dataset for Exploratory Data Analysis (EDA)

Output:

data/processed/cleaned_data.csv

In [1]:
# ============================================================
# Environment Check
# ============================================================

import sys
import platform

print("=" * 60)
print("Python Version :", sys.version)
print("Platform       :", platform.platform())
print("=" * 60)

Python Version : 3.14.4 (v3.14.4:23116f998f6, Apr  7 2026, 09:45:22) [Clang 17.0.0 (clang-1700.6.4.2)]
Platform       : macOS-26.5.2-arm64-arm-64bit-Mach-O


In [2]:
# ============================================================
# Import Required Libraries
# ============================================================

import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

print("Libraries Imported Successfully.")

Libraries Imported Successfully.


In [3]:
# ============================================================
# Project Directory Structure
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "Data"
RAW_DATA_DIR = DATA_DIR / "Raw"
PROCESSED_DATA_DIR = DATA_DIR / "Processed"
REPORT_DIR = PROJECT_ROOT / "Reports"
REPORT_DIR.mkdir(exist_ok=True)
PROCESSED_DATA_DIR.mkdir(exist_ok=True)

print("Project Root :", PROJECT_ROOT)
print("Raw Data     :", RAW_DATA_DIR)
print("Processed    :", PROCESSED_DATA_DIR)

Project Root : /Users/syedfaizaanahmad/Desktop/Project/AI-Based-Enviro-Pollution-Forecasting-Sys-for-Persona-Health-Alerts---Smart-Route-Plann-Jun-2026
Raw Data     : /Users/syedfaizaanahmad/Desktop/Project/AI-Based-Enviro-Pollution-Forecasting-Sys-for-Persona-Health-Alerts---Smart-Route-Plann-Jun-2026/Data/Raw
Processed    : /Users/syedfaizaanahmad/Desktop/Project/AI-Based-Enviro-Pollution-Forecasting-Sys-for-Persona-Health-Alerts---Smart-Route-Plann-Jun-2026/Data/Processed


## Dataset Discovery

The project automatically searches the raw data directory for all CSV files.

This allows new monthly datasets to be added without changing the notebook.

In [4]:
# ============================================================
# Locate Dataset Files
# ============================================================

csv_files = sorted(glob.glob(str(RAW_DATA_DIR / "**/*.csv"), recursive=True))
print(f"Total CSV Files Found : {len(csv_files)}")

for file in csv_files:
    print(Path(file).name)

Total CSV Files Found : 1
AQI_Dataset.csv


In [5]:
# Stop execution if no dataset is found

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files found inside:\n{RAW_DATA_DIR}"
    )

print("Dataset files detected successfully.")

Dataset files detected successfully.


## Load Raw Data

All CSV files are loaded one by one.

During loading:

- Source filename is recorded.
- Individual file shape is logged.
- DataFrames are stored temporarily.

In [6]:
# ============================================================
# Load All CSV Files
# ============================================================

dataframes = []
loading_summary = []
for file in csv_files:
    df = pd.read_csv(file)
    df["source_file"] = Path(file).name
    dataframes.append(df)
    loading_summary.append({"File": Path(file).name, "Rows": df.shape[0], "Columns": df.shape[1]})
print(f"Loaded {len(dataframes)} datasets.")

Loaded 1 datasets.


In [7]:
# Loading Summary
loading_report = pd.DataFrame(loading_summary)
display(loading_report)

,File,Rows,Columns
0,AQI_Dataset.csv,3431900,26


In [8]:
# ============================================================
# Merge All Datasets
# ============================================================

df = pd.concat(dataframes, ignore_index=True)
print("Merged Dataset Shape")
print(df.shape)

Merged Dataset Shape
(3431900, 26)


In [9]:
# Preview Dataset
display(df.head())

,Timestamp,PM2.5,PM10,Nitric Oxide,Nitrogen Dioxide,Nitrogen Oxides,Ammonia,Sulfur Dioxide,Carbon Monoxide,Ozone,Ambient Temperature,Relative Humidity,Solar Radiation,Rainfall,State,City,Latitude,Longitude,Calculated_AQI,Month,Hour,DayOfWeek,Is_Weekend,State_Encoded,City_Encoded,source_file
0,2017-09-05 11:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,11,1,0,0,3,AQI_Dataset.csv
1,2017-09-05 12:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,12,1,0,0,3,AQI_Dataset.csv
2,2017-09-05 13:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,13,1,0,0,3,AQI_Dataset.csv
3,2017-09-05 14:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,14,1,0,0,3,AQI_Dataset.csv
4,2017-09-05 15:00:00,23.000,49.500,0.650,14.550,8.280,8.850,4.520,0.150,62.500,32.220,70.500,290.750,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,15,1,0,0,3,AQI_Dataset.csv


In [10]:
# Dataset Information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3431900 entries, 0 to 3431899
Data columns (total 26 columns):
 #   Column               Dtype  
---  ------               -----  
 0   Timestamp            str    
 1   PM2.5                float64
 2   PM10                 float64
 3   Nitric Oxide         float64
 4   Nitrogen Dioxide     float64
 5   Nitrogen Oxides      float64
 6   Ammonia              float64
 7   Sulfur Dioxide       float64
 8   Carbon Monoxide      float64
 9   Ozone                float64
 10  Ambient Temperature  float64
 11  Relative Humidity    float64
 12  Solar Radiation      float64
 13  Rainfall             float64
 14  State                str    
 15  City                 str    
 16  Latitude             float64
 17  Longitude            float64
 18  Calculated_AQI       float64
 19  Month                int64  
 20  Hour                 int64  
 21  DayOfWeek            int64  
 22  Is_Weekend           int64  
 23  State_Encoded        int64  
 24  City_Enco

# Data Standardization

Different datasets often use different naming conventions.

Examples:

- PM2.5
- PM2_5
- pm25
- pm_2_5

Before cleaning the data, we standardize:

- Column names
- Data types
- Text formatting
- Datetime columns

This ensures that all subsequent notebooks work with a consistent schema.

In [11]:
# ============================================================
# Standardize Column Names
# ============================================================

def standardize_column_names(columns):
    standardized = []
    for col in columns:
        col = str(col).strip().lower()
        col = col.replace(" ", "_")
        col = col.replace("-", "_")
        col = col.replace("/", "_")
        col = col.replace("(", "")
        col = col.replace(")", "")
        col = col.replace(".", "")
        standardized.append(col)

    return standardized

df.columns = standardize_column_names(df.columns)
print("Column names standardized successfully.")

Column names standardized successfully.


In [12]:
# Display Updated Columns
print("Columns Present:\n")
for col in df.columns:
    print(col)

Columns Present:

timestamp
pm25
pm10
nitric_oxide
nitrogen_dioxide
nitrogen_oxides
ammonia
sulfur_dioxide
carbon_monoxide
ozone
ambient_temperature
relative_humidity
solar_radiation
rainfall
state
city
latitude
longitude
calculated_aqi
month
hour
dayofweek
is_weekend
state_encoded
city_encoded
source_file


# Remove Leading and Trailing Spaces

Text columns frequently contain unwanted spaces.

For example:

"Delhi "

and

" Delhi"

should both become

"Delhi"

In [13]:
# ============================================================
# Clean String Columns
# ============================================================

object_columns = df.select_dtypes(include="object").columns
for column in object_columns:
    df[column] = (df[column].astype(str).str.strip())
print(f"Processed {len(object_columns)} text columns.")

Processed 4 text columns.


In [14]:
# Preview Dataset After Cleaning Text
display(df.head())

,timestamp,pm25,pm10,nitric_oxide,nitrogen_dioxide,nitrogen_oxides,ammonia,sulfur_dioxide,carbon_monoxide,ozone,ambient_temperature,relative_humidity,solar_radiation,rainfall,state,city,latitude,longitude,calculated_aqi,month,hour,dayofweek,is_weekend,state_encoded,city_encoded,source_file
0,2017-09-05 11:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,11,1,0,0,3,AQI_Dataset.csv
1,2017-09-05 12:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,12,1,0,0,3,AQI_Dataset.csv
2,2017-09-05 13:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,13,1,0,0,3,AQI_Dataset.csv
3,2017-09-05 14:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,14,1,0,0,3,AQI_Dataset.csv
4,2017-09-05 15:00:00,23.000,49.500,0.650,14.550,8.280,8.850,4.520,0.150,62.500,32.220,70.500,290.750,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,15,1,0,0,3,AQI_Dataset.csv


# Basic Dataset Statistics

Before cleaning, we examine:

- Number of rows
- Number of columns
- Data types
- Missing values
- Memory usage

In [15]:
# ============================================================
# Dataset Summary
# ============================================================

summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing Values": df.isna().sum(),
    "Missing (%)": (
        df.isna().mean() * 100
    ).round(2),
    "Unique Values": df.nunique()
})

display(summary)

,Data Type,Missing Values,Missing (%),Unique Values
timestamp,str,0,0.000,83231
pm25,float64,0,0.000,185767
pm10,float64,0,0.000,245680
nitric_oxide,float64,0,0.000,139577
nitrogen_dioxide,float64,0,0.000,148978
nitrogen_oxides,float64,0,0.000,143254
ammonia,float64,0,0.000,137045
sulfur_dioxide,float64,0,0.000,136233
carbon_monoxide,float64,0,0.000,50461
ozone,float64,0,0.000,156733


In [16]:
# Shape of Dataset

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Rows    : 3,431,900
Columns : 26


# Memory Usage

Large environmental datasets often span several years and multiple cities.

Understanding memory usage helps optimize preprocessing.

In [17]:
# ============================================================
# Memory Usage
# ============================================================

memory_mb = (df.memory_usage(deep=True).sum() /(1024 ** 2))
print(f"Dataset Memory Usage : {memory_mb:.2f} MB")

Dataset Memory Usage : 909.08 MB


# Duplicate Record Detection

Duplicate observations may occur when:

- Multiple files overlap
- Data is downloaded more than once
- Sensor uploads are repeated

We first identify duplicates before removing them.

In [18]:
# ============================================================
# Detect Duplicate Records
# ============================================================

duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows : {duplicate_count:,}")

Duplicate Rows : 21


In [19]:
# Display Duplicate Rows (if any)

if duplicate_count > 0:
    display(df[df.duplicated()].head())
else:
    print("No duplicate records found.")

,timestamp,pm25,pm10,nitric_oxide,nitrogen_dioxide,nitrogen_oxides,ammonia,sulfur_dioxide,carbon_monoxide,ozone,ambient_temperature,relative_humidity,solar_radiation,rainfall,state,city,latitude,longitude,calculated_aqi,month,hour,dayofweek,is_weekend,state_encoded,city_encoded,source_file
1085533,2026-03-05 15:00:00,53.500,106.000,6.000,23.500,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,15,3,0,7,20,AQI_Dataset.csv
1085535,2026-03-05 16:00:00,49.000,95.500,6.000,38.500,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,16,3,0,7,20,AQI_Dataset.csv
1085537,2026-03-05 17:00:00,61.500,114.250,6.000,27.000,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,17,3,0,7,20,AQI_Dataset.csv
1085539,2026-03-05 18:00:00,94.000,175.250,6.000,93.500,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,18,3,0,7,20,AQI_Dataset.csv
1085541,2026-03-05 19:00:00,133.750,254.000,6.000,46.250,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,19,3,0,7,20,AQI_Dataset.csv


In [20]:
# ============================================================
# Remove Duplicate Records
# ============================================================
rows_before = len(df)
df = df.drop_duplicates(subset=["city", "timestamp"], keep="first").reset_index(drop=True)
rows_after = len(df)

print("Rows Before :", rows_before)
print("Rows After  :", rows_after)
print("Removed     :", rows_before - rows_after)

Rows Before : 3431900
Rows After  : 3429120
Removed     : 2780


# Datetime Processing

Time is the most important feature in environmental forecasting.

This section automatically:

- Detects datetime columns
- Converts them into pandas datetime format
- Creates a unified timestamp column
- Sorts the dataset chronologically
- Validates missing timestamps

In [21]:
# ============================================================
# Detect Possible Datetime Columns
# ============================================================
possible_datetime_columns = []
datetime_keywords = ["date", "time", "datetime", "timestamp"]

for column in df.columns:
    name = column.lower()
    if any(keyword in name for keyword in datetime_keywords):
        possible_datetime_columns.append(column)

print("Possible Datetime Columns:\n")
for column in possible_datetime_columns:
    print(f"• {column}")

Possible Datetime Columns:

• timestamp


In [22]:
# ============================================================
# Convert Datetime Columns
# ============================================================

for column in possible_datetime_columns:
    try:
        df[column] = pd.to_datetime(df[column], errors="coerce")
        print(f"Converted : {column}")

    except Exception as e:
        print(f"Skipped : {column}")

Converted : timestamp


In [23]:
# Preview Converted Columns

display(df[possible_datetime_columns].head())

,timestamp
0,2017-09-05 11:00:00
1,2017-09-05 12:00:00
2,2017-09-05 13:00:00
3,2017-09-05 14:00:00
4,2017-09-05 15:00:00


# Create Master Timestamp

Many datasets contain separate:

- Date
- Time

columns.

If a timestamp column does not already exist,
we create one automatically.

In [24]:
# ============================================================
# Create Timestamp Column
# ============================================================

if "timestamp" not in df.columns:
    if "datetime" in df.columns:
        df["timestamp"] = df["datetime"]
    elif "date" in df.columns and "time" in df.columns:
        df["timestamp"] = pd.to_datetime(df["date"].astype(str) + " " + df["time"].astype(str), errors="coerce")
    elif "date" in df.columns:
        df["timestamp"] = pd.to_datetime(df["date"], errors="coerce")

print("Timestamp column created.")

Timestamp column created.


In [25]:
# Sort Dataset Chronologically

if "timestamp" in df.columns:
    df = df.sort_values(by="timestamp").reset_index(drop=True)
print("Dataset sorted by timestamp.")

Dataset sorted by timestamp.


In [26]:
# Timestamp Summary

if "timestamp" in df.columns:

    print("Start :", df["timestamp"].min())
    print("End   :", df["timestamp"].max())
    print("\nMissing Timestamp Values:")
    print(df["timestamp"].isna().sum())

Start : 2017-01-01 00:00:00
End   : 2026-06-30 23:00:00

Missing Timestamp Values:
0


# Missing Value Analysis

Before deciding how to treat missing values,
we calculate the percentage of missing data
for every feature.

In [27]:
# ============================================================
# Missing Value Report
# ============================================================

missing_report = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)})
missing_report = missing_report.sort_values(by="Missing Percentage", ascending=False)
display(missing_report)

,Missing Values,Missing Percentage
timestamp,0,0.000
pm25,0,0.000
city_encoded,0,0.000
state_encoded,0,0.000
is_weekend,0,0.000
dayofweek,0,0.000
hour,0,0.000
month,0,0.000
calculated_aqi,0,0.000
longitude,0,0.000


In [28]:
# Columns With Missing Values

missing_columns = missing_report[missing_report["Missing Values"] > 0]
print(f"Columns containing missing values : {len(missing_columns)}")
display(missing_columns)

Columns containing missing values : 0


,Missing Values,Missing Percentage


# Missing Value Treatment Strategy

Different feature types require different approaches.

Numeric Features

- Median Imputation

Categorical Features

- Mode Imputation

Datetime Features

- Forward Fill

In [29]:
# ============================================================
# Fill Numeric Missing Values
# ============================================================

numeric_columns = df.select_dtypes(include=np.number).columns
for column in numeric_columns:
    median_value = df[column].median()
    df[column] = df[column].fillna(median_value)
print("Numeric missing values handled.")

Numeric missing values handled.


In [30]:
# ============================================================
# Fill Categorical & Datetime Missing Values
# ============================================================

categorical_columns = df.select_dtypes(include="object").columns
datetime_columns = df.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns
# Fill categorical columns with mode
for column in categorical_columns:
    if df[column].isna().sum() > 0:
        mode_value = df[column].mode(dropna=True)
        if not mode_value.empty:
            df[column] = df[column].fillna(mode_value.iloc[0])

# Fill datetime columns using forward fill
for column in datetime_columns:
    df[column] = df[column].ffill().bfill()

print("Categorical and datetime missing values handled.")

Categorical and datetime missing values handled.


# Data Validation

Before using the data for analysis or modeling, we perform several quality checks:

- Optimize data types to reduce memory usage
- Validate pollutant values
- Detect impossible measurements
- Handle outliers
- Generate a data quality report

This step ensures the dataset is reliable for downstream analysis.

In [31]:
# ============================================================
# Optimize Numeric Data Types
# ============================================================

memory_before = df.memory_usage(deep=True).sum() / (1024 ** 2)

for column in df.select_dtypes(include=["int64"]).columns:
    df[column] = pd.to_numeric(df[column], downcast="integer")

for column in df.select_dtypes(include=["float64"]).columns:
    df[column] = pd.to_numeric(df[column], downcast="float")

memory_after = df.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Memory Before : {memory_before:.2f} MB")
print(f"Memory After  : {memory_after:.2f} MB")
print(f"Memory Saved  : {memory_before-memory_after:.2f} MB")

Memory Before : 845.82 MB
Memory After  : 499.17 MB
Memory Saved  : 346.65 MB


In [32]:
# ============================================================
# Display Optimized Data Types
# ============================================================

display(df.dtypes)

timestamp              datetime64[us]
pm25                          float32
pm10                          float32
nitric_oxide                  float32
nitrogen_dioxide              float32
nitrogen_oxides               float32
ammonia                       float32
sulfur_dioxide                float32
carbon_monoxide               float32
ozone                         float32
ambient_temperature           float32
relative_humidity             float32
solar_radiation               float32
rainfall                      float32
state                             str
city                              str
latitude                      float32
longitude                     float32
calculated_aqi                float32
month                            int8
hour                             int8
dayofweek                        int8
is_weekend                       int8
state_encoded                    int8
city_encoded                     int8
source_file                       str
dtype: objec

# Detect Pollutant Columns

The notebook automatically identifies pollutant measurements.

Supported pollutants include:

- PM2.5
- PM10
- NO
- NO₂
- NOx
- SO₂
- CO
- O₃
- NH₃
- Benzene
- Toluene
- Xylene
- AQI

In [33]:
# ============================================================
# Detect Pollution Columns (Refined)
# ============================================================

# Removed "co" and "no" to prevent matching "encoded" or other words
pollutant_keywords = [
    "pm",
    "aqi",
    "no2",
    "nox",
    "nitric",       
    "nitrogen",     
    "sulfur",       
    "carbon",       
    "ozone",        
    "ammonia",      
    "so2",
    "o3",
    "nh3",
    "benzene",
    "toluene",
    "xylene"
]

pollutant_columns = []

for column in df.columns:
    column_lower = column.lower()
    
    if any(keyword in column_lower for keyword in pollutant_keywords):
        if "encoded" not in column_lower:
            pollutant_columns.append(column)

print("Detected Pollutant Columns:\n")

for col in pollutant_columns:
    print(f"• {col}")

Detected Pollutant Columns:

• pm25
• pm10
• nitric_oxide
• nitrogen_dioxide
• nitrogen_oxides
• ammonia
• sulfur_dioxide
• carbon_monoxide
• ozone
• calculated_aqi


In [34]:
# Pollutant Summary Statistics

if pollutant_columns:

    display(df[pollutant_columns].describe().T)
else:
    print("No pollutant columns detected.")

,count,mean,std,min,25%,50%,75%,max
pm25,3429120.000,53.907,60.351,0.000,21.000,36.165,64.250,999.990
pm10,3429120.000,114.714,106.505,0.000,48.688,82.750,142.070,1000.000
nitric_oxide,3429120.000,13.622,30.184,0.000,2.780,6.000,12.325,500.000
nitrogen_dioxide,3429120.000,24.424,27.293,0.000,8.860,16.370,29.160,499.990
nitrogen_oxides,3429120.000,28.205,36.784,0.000,10.400,19.000,32.650,500.000
ammonia,3429120.000,21.168,22.207,0.000,8.500,15.991,25.630,499.990
sulfur_dioxide,3429120.000,13.057,15.727,0.000,4.850,8.820,15.500,200.000
carbon_monoxide,3429120.000,0.833,0.841,0.000,0.360,0.640,1.015,41.670
ozone,3429120.000,27.637,27.458,0.000,9.590,19.290,35.910,932.000
calculated_aqi,3429120.000,118.121,90.928,0.122,53.902,90.106,145.535,407.693


# Invalid Value Detection

Environmental measurements cannot be negative.

Examples:

PM2.5 = -10 ❌

CO = -1 ❌

Such values are treated as invalid and replaced with missing values.

In [35]:
# ============================================================
# Replace Invalid Negative Values
# ============================================================

invalid_value_summary = []

for column in pollutant_columns:
    negative_count = (df[column] < 0).sum()
    invalid_value_summary.append({"Column": column, "Negative Values": int(negative_count)})
    df.loc[df[column] < 0, column] = np.nan

invalid_summary = pd.DataFrame(invalid_value_summary)
display(invalid_summary)

,Column,Negative Values
0,pm25,0
1,pm10,0
2,nitric_oxide,0
3,nitrogen_dioxide,0
4,nitrogen_oxides,0
5,ammonia,0
6,sulfur_dioxide,0
7,carbon_monoxide,0
8,ozone,0
9,calculated_aqi,0


In [36]:
# Fill Newly Created Missing Values

for column in pollutant_columns:
    df[column] = (df[column].ffill().bfill().fillna(df[column].median()))

print("Negative values handled successfully.")

Negative values handled successfully.


# Outlier Detection

Extreme pollution values may occur because of:

- Sensor malfunction
- Recording errors
- Exceptional environmental events

Rather than deleting observations, we cap extreme values using the IQR method.

In [37]:
# ============================================================
# Detect Outliers Using IQR
# ============================================================

outlier_summary = []
for column in pollutant_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[column] < lower) | (df[column] > upper)).sum()
    outlier_summary.append({"Column": column, "Outliers": int(outliers)})

outlier_report = pd.DataFrame(outlier_summary)
display(outlier_report)

,Column,Outliers
0,pm25,255238
1,pm10,227254
2,nitric_oxide,336979
3,nitrogen_dioxide,266214
4,nitrogen_oxides,245447
5,ammonia,216489
6,sulfur_dioxide,249677
7,carbon_monoxide,228567
8,ozone,224267
9,calculated_aqi,316042


In [38]:
# ============================================================
# Cap Outliers (Winsorization using IQR)
# ============================================================

for column in pollutant_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[column] = df[column].clip(lower, upper)

print("Outliers capped successfully.")

Outliers capped successfully.


In [39]:
# Verify Missing Values

missing_after_cleaning = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Percentage": (df.isna().mean() * 100).round(2)})

display(missing_after_cleaning)

,Missing Values,Percentage
timestamp,0,0.000
pm25,0,0.000
pm10,0,0.000
nitric_oxide,0,0.000
nitrogen_dioxide,0,0.000
nitrogen_oxides,0,0.000
ammonia,0,0.000
sulfur_dioxide,0,0.000
carbon_monoxide,0,0.000
ozone,0,0.000


In [40]:
# ============================================================
# Data Quality Report
# ============================================================

quality_report = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Duplicate Rows", "Missing Cells", "Memory Usage (MB)"],

    "Value": [
        len(df),
        df.shape[1],
        df.duplicated().sum(),
        df.isna().sum().sum(),
        round(df.memory_usage(deep=True).sum() / (1024**2), 2)]})

display(quality_report)

,Metric,Value
0,Rows,3429120.000
1,Columns,26.000
2,Duplicate Rows,0.000
3,Missing Cells,0.000
4,Memory Usage (MB),577.660


# Final Dataset Validation

Before exporting the cleaned dataset, we perform one last validation to ensure:

- No duplicate rows remain
- Required columns exist
- Timestamp column is valid
- Column names are standardized
- Dataset is sorted chronologically

The exported dataset becomes the single source of truth for the next notebook.

In [41]:
# ============================================================
# Final Dataset Shape
# ============================================================

print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)
print(f"Rows           : {len(df):,}")
print(f"Columns        : {df.shape[1]}")
print(f"Missing Values : {df.isna().sum().sum():,}")
print(f"Duplicate Rows : {df.duplicated().sum():,}")

FINAL DATASET SUMMARY
Rows           : 3,429,120
Columns        : 26
Missing Values : 0
Duplicate Rows : 0


In [42]:
# ============================================================
# Verify Timestamp
# ============================================================

if "timestamp" in df.columns:
    print("Timestamp column found.")
    print("Start :", df["timestamp"].min())
    print("End   :", df["timestamp"].max())
else:

    print("WARNING: Timestamp column not found.")

Timestamp column found.
Start : 2017-01-01 00:00:00
End   : 2026-06-30 23:00:00


In [43]:
# ============================================================
# Sort Dataset
# ============================================================

if "timestamp" in df.columns:
    df = (df.sort_values("timestamp").reset_index(drop=True))

print("Dataset sorted successfully.")

Dataset sorted successfully.


# Save Clean Dataset

The cleaned dataset is exported in two formats:

- CSV → Easy to inspect manually
- Parquet → Faster loading and smaller storage

Subsequent notebooks will load the Parquet file by default.

In [44]:
# ============================================================
# Output File Paths
# ============================================================

clean_csv = PROCESSED_DATA_DIR / "cleaned_data.csv"
print(clean_csv)

/Users/syedfaizaanahmad/Desktop/Project/AI-Based-Enviro-Pollution-Forecasting-Sys-for-Persona-Health-Alerts---Smart-Route-Plann-Jun-2026/Data/Processed/cleaned_data.csv


In [45]:
# ============================================================
# Save Clean Dataset
# ============================================================

df.to_csv(clean_csv, index=False)
print("Clean dataset saved successfully.")

Clean dataset saved successfully.


In [46]:
# ============================================================
# Create Dataset Metadata
# ============================================================

metadata = {
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "missing_values": int(df.isna().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "memory_mb": round(df.memory_usage(deep=True).sum() / (1024**2), 2),
    "columns_list": list(df.columns)
}

metadata

{'rows': 3429120,
 'columns': 26,
 'missing_values': 0,
 'duplicate_rows': 0,
 'memory_mb': np.float64(577.66),
 'columns_list': ['timestamp',
  'pm25',
  'pm10',
  'nitric_oxide',
  'nitrogen_dioxide',
  'nitrogen_oxides',
  'ammonia',
  'sulfur_dioxide',
  'carbon_monoxide',
  'ozone',
  'ambient_temperature',
  'relative_humidity',
  'solar_radiation',
  'rainfall',
  'state',
  'city',
  'latitude',
  'longitude',
  'calculated_aqi',
  'month',
  'hour',
  'dayofweek',
  'is_weekend',
  'state_encoded',
  'city_encoded',
  'source_file']}

In [47]:
# ============================================================
# Display Final Preview
# ============================================================
display(df.head())
display(df.tail())

,timestamp,pm25,pm10,nitric_oxide,nitrogen_dioxide,nitrogen_oxides,ammonia,sulfur_dioxide,carbon_monoxide,ozone,ambient_temperature,relative_humidity,solar_radiation,rainfall,state,city,latitude,longitude,calculated_aqi,month,hour,dayofweek,is_weekend,state_encoded,city_encoded,source_file
0,2017-01-01 00:00:00,129.125,227.000,26.642,59.610,66.025,51.325,21.520,1.950,8.330,14.500,71.830,12.250,0.000,Delhi,"Anand Vihar ( , )",28.651,77.315,90.106,1,0,6,1,6,4,AQI_Dataset.csv
1,2017-01-01 00:00:00,129.125,119.490,26.642,55.510,29.890,15.991,18.890,1.997,0.710,21.810,19.880,0.700,0.000,Gujarat,"Maninagar, Ahmedabad ( , )",22.996,72.600,90.106,1,0,6,1,7,38,AQI_Dataset.csv
2,2017-01-01 01:00:00,129.125,119.490,20.770,53.560,29.890,15.991,25.140,1.700,1.287,20.990,29.470,0.700,0.000,Gujarat,"Maninagar, Ahmedabad ( , )",22.996,72.600,90.106,1,1,6,1,7,38,AQI_Dataset.csv
3,2017-01-01 01:00:00,129.125,226.170,26.642,59.610,66.025,51.325,21.760,1.997,8.980,13.660,73.250,12.170,0.000,Delhi,"Anand Vihar ( , )",28.651,77.315,90.106,1,1,6,1,6,4,AQI_Dataset.csv
4,2017-01-01 02:00:00,125.980,119.490,24.290,51.040,29.890,15.991,30.700,1.530,1.863,20.080,42.360,0.700,0.000,Gujarat,"Maninagar, Ahmedabad ( , )",22.996,72.600,90.106,1,2,6,1,7,38,AQI_Dataset.csv


,timestamp,pm25,pm10,nitric_oxide,nitrogen_dioxide,nitrogen_oxides,ammonia,sulfur_dioxide,carbon_monoxide,ozone,ambient_temperature,relative_humidity,solar_radiation,rainfall,state,city,latitude,longitude,calculated_aqi,month,hour,dayofweek,is_weekend,state_encoded,city_encoded,source_file
3429115,2026-06-30 23:00:00,26.800,37.500,26.642,40.200,61.400,32.690,5.240,0.670,19.440,24.970,87.280,17.200,0.000,West Bengal,"Ballygunge, Kolkata (, )",22.529,88.362,38.991,6,23,1,0,18,10,AQI_Dataset.csv
3429116,2026-06-30 23:00:00,32.000,131.000,3.970,52.530,31.170,12.370,1.430,1.510,7.530,30.170,74.000,14.000,0.000,Andhra Pradesh,"GVM Corporation, Visakhapatnam (, )",17.720,83.300,114.479,6,23,1,0,0,21,AQI_Dataset.csv
3429117,2026-06-30 23:00:00,19.720,38.800,12.610,24.800,22.660,12.600,9.430,0.490,24.420,26.200,99.550,41.100,0.000,West Bengal,"Asansol Court Area, Asansol (, )",23.685,86.946,42.429,6,23,1,0,18,6,AQI_Dataset.csv
3429118,2026-06-30 23:00:00,66.460,19.090,11.410,4.870,17.270,17.560,12.410,0.150,6.120,26.760,85.970,0.000,0.000,Uttar Pradesh,"Shivaji Nagar, Jhansi (, )",25.451,78.595,35.718,6,23,1,0,17,56,AQI_Dataset.csv
3429119,2026-06-30 23:00:00,27.740,93.610,20.990,59.610,66.025,51.325,10.850,1.100,23.700,26.580,97.620,14.370,0.000,West Bengal,"Ward-32 Bapupara, Siliguri (, )",26.732,88.410,36.862,6,23,1,0,18,65,AQI_Dataset.csv


In [48]:
# ============================================================
# Final Data Types
# ============================================================

display(df.dtypes)

timestamp              datetime64[us]
pm25                          float32
pm10                          float32
nitric_oxide                  float64
nitrogen_dioxide              float64
nitrogen_oxides               float64
ammonia                       float64
sulfur_dioxide                float64
carbon_monoxide               float32
ozone                         float32
ambient_temperature           float32
relative_humidity             float32
solar_radiation               float32
rainfall                      float32
state                             str
city                              str
latitude                      float32
longitude                     float32
calculated_aqi                float64
month                            int8
hour                             int8
dayofweek                        int8
is_weekend                       int8
state_encoded                    int8
city_encoded                     int8
source_file                       str
dtype: objec

In [49]:
# ============================================================
# Notebook Completion
# ============================================================

print("=" * 60)
print("01_data_preparation.ipynb COMPLETED SUCCESSFULLY")
print("=" * 60)
print()
print("Generated Files")
print("---------------------------")
print("✓ cleaned_data.csv")

print()
print("Next Notebook")
print("---------------------------")
print("02_EDA.ipynb")

01_data_preparation.ipynb COMPLETED SUCCESSFULLY

Generated Files
---------------------------
✓ cleaned_data.csv

Next Notebook
---------------------------
02_EDA.ipynb
